In [0]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq
import builtins

def xirr_brentq(dates, amounts):
    dates = pd.to_datetime(dates)
    t0 = dates.min()
    days = (dates - t0).days
    amounts = np.array(amounts, dtype=float)
    def xnpv(rate):
        return np.sum(amounts / (1 + rate) ** (days / 365))
    try:
        return brentq(xnpv, -0.9999, 10)
    except Exception:
        return np.nan

df = spark.sql("SELECT * FROM investment_vision.raw_data_xirr").toPandas()

def build_cashflow(group):
    cashflows = []
    for _, row in group.iterrows():
        for col in ['buy_amount', 'sell_amount', 'market_value']:
            if row[col] != 0:
                cashflows.append((row['date'], float(row[col])))
    if not cashflows:
        return [], []
    cashflows.sort(key=lambda x: x[0])
    dates, amounts = zip(*cashflows)
    return dates, amounts

results = []
for inst, group in df.groupby("Instrument"):
    dates, amounts = build_cashflow(group)
    if amounts:
        xirr_val = xirr_brentq(dates, amounts)
        total_buy = group['buy_amount'].sum()
        total_sell = group['sell_amount'].sum()
        market_value = group['market_value'].sum()
        holding_period = (pd.to_datetime(max(dates)) - pd.to_datetime(min(dates))).days
    else:
        xirr_val = np.nan
        total_buy = np.nan
        total_sell = np.nan
        market_value = np.nan
        holding_period = np.nan
    if isinstance(xirr_val, (int, float, np.floating)) and np.isfinite(xirr_val):
        xirr_display = builtins.round(float(xirr_val) * 100, 2)
    else:
        xirr_display = np.nan
    results.append({
        "Instrument": inst,
        "XIRR (%)": xirr_display,
        "Total Buy Amount": builtins.round(total_buy, 2) if isinstance(total_buy, (int, float, np.floating)) and np.isfinite(total_buy) else np.nan,
        "Total Sell Amount": builtins.round(total_sell, 2) if isinstance(total_sell, (int, float, np.floating)) and np.isfinite(total_sell) else np.nan,
        "Market Value": builtins.round(market_value, 2) if isinstance(market_value, (int, float, np.floating)) and np.isfinite(market_value) else np.nan,
        "Holding Period (days)": holding_period
    })
results_df = pd.DataFrame(results)
display(results_df)


In [0]:
# Rename columns to valid Delta names before saving
spark_df = spark.createDataFrame(results_df.rename(columns={
    'XIRR (%)': 'XIRR_pct',
    'Total Buy Amount': 'Total_Buy_Amount',
    'Total Sell Amount': 'Total_Sell_Amount',
    'Market Value': 'Market_Value',
    'Holding Period (days)': 'Holding_Period_days'
}))
display(spark_df)
spark_df.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable('workspace.investment_vision.silver_xirr_details')